<a href="https://colab.research.google.com/github/prasanna-venkatesh-m/tiny-transformer/blob/main/Tiny_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install the Gensim and NLTK

In [9]:
pip install gensim nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 75.3 MB/s eta 0:00:00


Import and download all the Packages

In [66]:
import gensim
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.tokenize import word_tokenize
import tensorflow as tf

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Converting the sentance into token (Like spliting the words)

In [78]:
sentances = "The cat sat on the mat"
# converting the sentance into tokens (by words or subwords)
tokens = word_tokenize(sentances.lower(), language='english', preserve_line=True)
print(tokens)

# converting the tokens into sets
vocab = {}
i =0
for word in set(tokens):
  vocab[word]=i
  i = i+1
print(vocab)

vocab_ids = []
for word in tokens:
  vocab_ids.append(vocab[word])
print(vocab_ids)

['the', 'cat', 'sat', 'on', 'the', 'mat']
{'the': 0, 'mat': 1, 'sat': 2, 'on': 3, 'cat': 4}
[0, 4, 2, 3, 0, 1]


Convert it to Tensors

In [82]:
# converting the normal array to Tensor
ids_tensor = tf.constant(vocab_ids)

# vector embedding (for embedding vector layer)
seq_len = len(vocab_ids)
embedding_dim = 64
embedding_layer = tf.keras.layers.Embedding(input_dim = seq_len, output_dim = embedding_dim)

# convert the tokens to vector embeddings
token_embeddings = embedding_layer(ids_tensor)
print(token_embeddings.shape)

(6, 64)


# Positional Embeddings

In [97]:
# Initialize the pos_encoding with zeros (token_size, embedding_dimension)
pos_encoding = np.zeros((seq_len, embedding_dim))

# fill the pos_encoding
for i in range(seq_len):
  for j in range(embedding_dim):
    if i % 2 == 0:
      pos_encoding[i,j] = np.sin(i/(10000 ** (j / embedding_dim)))
    else:
      pos_encoding[i,j] = np.cos(i/ (10000 ** ((j-1)/embedding_dim)))

#convert to tensor
pos_encoding = tf.cast(pos_encoding, dtype=tf.float32)

# print pos encoding
print(pos_encoding.shape)

(6, 64)


# Adding the Token Embeddings and Positional Encoding to get Final Input

In [100]:
final_input = tf.add(token_embeddings , pos_encoding)
print(final_input.shape)

(6, 64)


# Self Attention

In [107]:
# we can initalize the dimension we want to set to Weights of Q,K,V
d = 5

# assign random weights to the Q,K,V
wQ = tf.Variable(tf.random.normal([embedding_dim, d]))
wK = tf.Variable(tf.random.normal([embedding_dim, d]))
wV = tf.Variable(tf.random.normal([embedding_dim, d]))

# multiply the final input with Weights to get Q,K,V
Q = tf.matmul(final_input, wQ)
K = tf.matmul(final_input, wK)
V = tf.matmul(final_input, wV)

print(Q.shape)
print(K.shape)
print(V.shape)

(6, 5)
(6, 5)
(6, 5)


**Scaled Dot-Product Attention**

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V
$$

Where:  
- \(Q\) = Query matrix  
- \(K\) = Key matrix  
- \(V\) = Value matrix  
- \(d_k\) = dimension of the keys  
- softmax = applied row-wise to get attention weights


In [118]:
# calculate the attention_score by multiplying Q.K^T
attention_scores = tf.matmul(Q, tf.transpose(K))

# divide the attention score by squareroot of dk
attention_scores = attention_scores / tf.sqrt(tf.cast(len(K), dtype=tf.float32))
print(f"Attention score shape : {attention_scores.shape}")

# take the softmax of the attentions scores
attention_weights = tf.nn.softmax(attention_scores, axis=-1)
print(f"Attention weights shape : {attention_weights.shape}")

# multiply with V to get the attention output
attention_output = tf.matmul(attention_weights, V)
print(f"Attention output shape : {attention_output.shape}")

Attention score shape : (6, 6)
Attention weights shape : (6, 6)
Attention output shape : (6, 5)
